In [ ]:
from helpers import *
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def aggregate_gpu_data_new(year: str) -> pd.DataFrame:
    """
    Aggregates GPU usage data for all months in a given year by merging job records with node statistics.

    Parameters:
        year (str): Two-digit year string (e.g., "25" for 2025).

    Returns:
        pd.DataFrame: A merged DataFrame containing job and GPU usage records for the entire year.
    """
    # Validate year format
    if not isinstance(year, str) or not year.isdigit() or len(year) != 2:
        raise ValueError(
            f"Invalid year format: {year}. Expected a two-digit string (e.g., '25' for 2025)."
        )

    # Load job records for the entire year
    gpu_jobs = read_gpu_records(f"/projectnb/rcsmetrics/accounting/data/scc/20{year}.csv")
    gpu_jobs["task_string"] = gpu_jobs["task_number"].astype(str)
    gpu_jobs.loc[~(gpu_jobs["options"].str.contains("-t")), "task_string"] = "undefined"
    gpu_jobs["job_task"] = (
        gpu_jobs["job_number"].astype(str) + "." + gpu_jobs["task_string"].astype(str)
    )

    # Get all files for the entire year
    nodes = os.listdir("/project/scv/dugan/gpustats/data/")
    files = []
    for month in [f"{m:02d}" for m in range(1, 13)]:  # Generate "01" to "12"
        files.extend([
            f"/project/scv/dugan/gpustats/data/{node}/{year}{month}"
            for node in nodes
            if os.path.exists(f"/project/scv/dugan/gpustats/data/{node}/{year}{month}")
        ])

    # Process and merge data
    all_merged_dfs = []
    for file_name in files:
        try:
            gpu_records = pd.DataFrame(clean_gpu_data(file_name))
        except Exception as e:
            print(f"Skipping missing or corrupted file: {file_name}")
            continue

        # Merge with job records
        merged_df = pd.merge(
            gpu_records, gpu_jobs, left_on="job_id", right_on="job_task", how="left"
        )
        all_merged_dfs.append(merged_df)

    # Return the final concatenated DataFrame
    return pd.concat(all_merged_dfs, ignore_index=True) if all_merged_dfs else pd.DataFrame()

In [ ]:
year_test = aggregate_gpu_data_new('24')
year_test

In [ ]:
year_test_old = aggregate_gpu_data('24')


In [ ]:
year_test.equals(year_test_old)

In [ ]:
jan = clean_gpu_data(f"/project/scv/dugan/gpustats/data/scc-x05/2401")
feb = clean_gpu_data(f"/project/scv/dugan/gpustats/data/scc-x05/2402")
jan.tail(10)

In [ ]:
feb.head(10)

In [ ]:
year_test_old['job_id'].str.contains('4197929.230').any()

In [ ]:
year_test_old[year_test_old['job_id']=='4197929.230']

In [ ]:
year_test_old['job_id'].str.contains('4198041.undefined').any()

In [ ]:
year_test_old[year_test_old['job_id']=='4198041.undefined']

## Iterating over files rather than monthly is equal but slower, cross month jobs are still present

In [ ]:
# test process_gpu_data_range(start_date: str, end_date: str)

ranged = process_gpu_data_range("2024-01-01","2024-12-31")
ranged.equals(year_test_old)

In [ ]:
# check jobs that cross months for monthly, or jobs that cross years for yearly

In [1]:
from userusage import *
import pandas as pd

In [2]:
df = read_user_records("2023-12-15", "2025-02-10", "ktrn")#[['time','owner','job_number']]
df#.columns

,qname,hostname,owner,job_name,job_number,ux_submission_time,ux_start_time,ux_end_time,failed,exit_status,...,project,granted_pe,slots,task_number,cpu,options,pe_taskid,maxvmem,n_gpu,time
1183,linga,scc-ve3,ktrn,rjob,3028153,1702928876,1702928942,1702928948,0,0,...,krcs,NaN,1,0,1.969588e+00,"-U sequencing -u ktrn -l h_rt=43200,no_gpu=TRU...",NaN,379199488,NaN,2023-12-18 19:49:08
1184,linga,scc-ve2,ktrn,lme2.INVRESWCHGT.19-Dec-2023_10-47.qsub,3061819,1703000873,1703000934,1703001259,0,0,...,krcs,NaN,1,12,3.239983e+02,"-U sequencing,cupples,linga -u ktrn -l cpu_arc...",NaN,511827968,NaN,2023-12-19 15:54:19
1185,linga,scc-gn3,ktrn,lme2.INVRESWCHGT.19-Dec-2023_10-47.qsub,3061819,1703000873,1703000929,1703001293,0,0,...,krcs,NaN,1,9,3.618717e+02,"-U sequencing,cupples,linga -u ktrn -l cpu_arc...",NaN,581443584,NaN,2023-12-19 15:54:53
1186,linga,scc-vd2,ktrn,lme2.INVRESWCHGT.19-Dec-2023_10-47.qsub,3061819,1703000873,1703000929,1703001295,0,0,...,krcs,NaN,1,11,3.653345e+02,"-U sequencing,cupples,linga -u ktrn -l cpu_arc...",NaN,564985856,NaN,2023-12-19 15:54:55
1187,linga,scc-vd2,ktrn,lme2.INVRESWCHGT.19-Dec-2023_10-47.qsub,3061819,1703000873,1703000929,1703001296,0,0,...,krcs,NaN,1,5,3.664112e+02,"-U sequencing,cupples,linga -u ktrn -l cpu_arc...",NaN,581087232,NaN,2023-12-19 15:54:56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2182,linga,scc-vd4,ktrn,jobs_tasks,2465919,1738613093,1738613170,1738613171,0,0,...,krcs,NaN,1,3,7.570030e-01,"-u ktrn -l h_rt=43200,no_gpu=TRUE -soft -l buy...",NaN,0,NaN,2025-02-03 20:06:11
2183,linga,scc-vd4,ktrn,jobs_tasks,2465919,1738613093,1738613170,1738613171,0,0,...,krcs,NaN,1,5,6.897980e-01,"-u ktrn -l h_rt=43200,no_gpu=TRUE -soft -l buy...",NaN,0,NaN,2025-02-03 20:06:11
2184,mem384,scc-yj3,ktrn,thread_4,2454149,1738540733,1738542632,1738821090,0,0,...,krcs,omp28,28,0,1.099957e+06,"-U geo,onrcc,uk-biobank,ukbbdata,bs859,academi...",NaN,106981945344,NaN,2025-02-06 05:51:30
2185,mem384,scc-yk4,ktrn,thread_8,2454150,1738540737,1738542988,1738828550,0,0,...,krcs,omp28,28,0,1.100930e+06,"-U geo,onrcc,uk-biobank,ukbbdata,bs859,academi...",NaN,108120862720,NaN,2025-02-06 07:55:50


In [3]:
df.columns

Index(['qname', 'hostname', 'owner', 'job_name', 'job_number',
       'ux_submission_time', 'ux_start_time', 'ux_end_time', 'failed',
       'exit_status', 'ru_wallclock', 'ru_utime', 'ru_maxrss', 'project',
       'granted_pe', 'slots', 'task_number', 'cpu', 'options', 'pe_taskid',
       'maxvmem', 'n_gpu', 'time'],
      dtype='object')